# Practice 07 — классификация заболеваний сердца

## Цель работы

Решить задачу бинарной классификации признака `target` на датасете `heart.csv`:

- самостоятельно реализовать метод K-ближайших соседей (KNN) с настраиваемыми гиперпараметрами;
- выполнить Data Cleaning и EDA;
- выполнить Feature Engineering;
- подобрать гиперпараметры с помощью 5-fold кросс-валидации;
- сравнить собственную реализацию с библиотечными `LogisticRegression`, `SVC`, `KNeighborsClassifier` и `DecisionTreeClassifier`;
- оценить модели на заранее отложенной тестовой выборке;
- построить confusion matrices и сделать выводы.

> Тестовая выборка используется только для финальной оценки. Подбор гиперпараметров выполняется исключительно внутри обучающей части посредством кросс-валидации.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

data_candidates = [
    Path("heart.csv"),
    Path("../heart.csv"),
    Path("../../data/heart_disease/heart.csv"),
]

DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден heart.csv. Положите файл рядом с notebook или измените DATA_PATH."
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Файл: {DATA_PATH}")
print(f"Размер данных: {df_raw.shape}")
display(df_raw.head())


## 1. Первичный анализ данных

Признаки:

- `age` — возраст;
- `sex` — пол;
- `cp` — тип боли в груди;
- `trestbps` — давление в покое;
- `chol` — холестерин;
- `fbs` — fasting blood sugar;
- `restecg` — результат ЭКГ;
- `thalach` — максимальная частота сердцебиения;
- `exang` — стенокардия при нагрузке;
- `oldpeak` — депрессия ST;
- `slope` — наклон пикового сегмента ST;
- `ca` — число крупных сосудов;
- `thal` — категория `thal`;
- `target` — наличие заболевания: `1` — есть, `0` — нет.


In [ ]:
print("Размер:", df_raw.shape)
print("\nТипы данных:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nПропуски:")
display(df_raw.isna().sum().to_frame("missing"))

print("\nОписательная статистика:")
display(df_raw.describe().T)

print("\nРаспределение target:")
display(df_raw["target"].value_counts().sort_index())

print("\nКоличество полных дубликатов:", df_raw.duplicated().sum())


### Data Cleaning

В исходном файле отсутствуют пропуски, однако обнаруживается большое количество полных дубликатов.

Дубликаты могут привести к утечке между train и test: одинаковые наблюдения способны попасть в обе выборки и искусственно завысить качество. Поэтому полные дубликаты удаляются **до** разбиения данных.


In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)

print(f"До очистки:    {df_raw.shape[0]} строк")
print(f"После очистки: {df.shape[0]} строк")
print(f"Удалено дубликатов: {df_raw.shape[0] - df.shape[0]}")

print("\nПропуски после очистки:")
display(df.isna().sum().to_frame("missing"))

print("\nБаланс классов:")
display(df["target"].value_counts().sort_index().to_frame("count"))

print("\nДоли классов:")
display(df["target"].value_counts(normalize=True).sort_index().to_frame("share"))


## 2. EDA

Исследуем баланс классов, численные признаки, категориальные признаки и корреляции с `target`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="target", ax=axes[0])
axes[0].set_title("Распределение target")
axes[0].set_xlabel("target (0 — нет, 1 — есть заболевание)")
axes[0].set_ylabel("Количество")

sns.histplot(
    data=df, x="age", hue="target",
    bins=15, kde=True, element="step", ax=axes[1]
)
axes[1].set_title("Возраст и наличие заболевания")
axes[1].set_xlabel("Возраст")

plt.tight_layout()
plt.show()


In [ ]:
print("Средние значения численных признаков по классам:")
display(
    df.groupby("target")[["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]]
      .mean()
      .T
)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.boxplot(data=df, x="target", y="age", ax=axes[0, 0])
axes[0, 0].set_title("Возраст")

sns.boxplot(data=df, x="target", y="thalach", ax=axes[0, 1])
axes[0, 1].set_title("Максимальная ЧСС")

sns.boxplot(data=df, x="target", y="oldpeak", ax=axes[1, 0])
axes[1, 0].set_title("Oldpeak")

sns.boxplot(data=df, x="target", y="trestbps", ax=axes[1, 1])
axes[1, 1].set_title("Давление в покое")

plt.tight_layout()
plt.show()


In [ ]:
categorical_eda = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
axes = axes.ravel()

for ax, col in zip(axes, categorical_eda):
    table = pd.crosstab(df[col], df["target"], normalize="index")
    table.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"{col} vs target")
    ax.set_xlabel(col)
    ax.set_ylabel("Доля")
    ax.legend(title="target", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица")
plt.tight_layout()
plt.show()

print("Корреляция признаков с target:")
display(corr["target"].sort_values(ascending=False).to_frame("corr_with_target"))


### Выводы EDA

1. После удаления дубликатов классы остаются достаточно сбалансированными.
2. Наиболее заметная связь с `target` наблюдается у `cp`, `thalach`, `oldpeak`, `exang`, `ca`, `thal` и `slope`.
3. Признаки имеют разные масштабы, поэтому для Logistic Regression, SVM и KNN требуется масштабирование.
4. Категориальные признаки разумно кодировать через one-hot encoding, чтобы не задавать им искусственную метрическую структуру.
5. Эти наблюдения используются при построении общего preprocessing pipeline.


## 3. Feature Engineering

Численные признаки:

`age`, `trestbps`, `chol`, `thalach`, `oldpeak`.

Категориальные признаки:

`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`.

Для численных признаков применяется `StandardScaler`, для категориальных — `OneHotEncoder`.

Преобразования помещены в `ColumnTransformer` и затем в `Pipeline`, поэтому при cross-validation статистики стандартизации и параметры кодирования не вычисляются по validation/test данным.


In [ ]:
TARGET = "target"

numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features),
    ]
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


# 4. Самостоятельная реализация KNN

Для нового объекта находятся `k` ближайших объектов обучающей выборки.

Настраиваемые гиперпараметры:

- `n_neighbors` — число соседей;
- `p` — степень расстояния Минковского;
- `weights` — равные или обратно пропорциональные расстоянию веса.

Расстояние Минковского:

\[
d_p(x,z)=\left(\sum_j |x_j-z_j|^p\right)^{1/p}.
\]

При `p=1` получаем манхэттенское расстояние, при `p=2` — евклидово.


In [ ]:
class MyKNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, p=2, weights="uniform"):
        self.n_neighbors = n_neighbors
        self.p = p
        self.weights = weights

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2:
            raise ValueError("X должен быть двумерным.")
        if len(X) != len(y):
            raise ValueError("X и y должны иметь одинаковую длину.")
        if self.weights not in ("uniform", "distance"):
            raise ValueError("weights должен быть 'uniform' или 'distance'.")
        if self.n_neighbors < 1:
            raise ValueError("n_neighbors должен быть >= 1.")
        if self.p < 1:
            raise ValueError("p должен быть >= 1.")

        self.X_train_ = X
        self.y_train_ = y
        self.classes_ = np.unique(y)
        return self

    def _distance(self, X):
        diff = X[:, None, :] - self.X_train_[None, :, :]
        return np.sum(np.abs(diff) ** self.p, axis=2) ** (1.0 / self.p)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        distances = self._distance(X)

        k = min(self.n_neighbors, len(self.X_train_))
        nearest_idx = np.argpartition(
            distances, kth=k - 1, axis=1
        )[:, :k]

        predictions = []

        for row_idx, neighbors in enumerate(nearest_idx):
            labels = self.y_train_[neighbors]

            if self.weights == "uniform":
                votes = {
                    cls: np.sum(labels == cls)
                    for cls in self.classes_
                }
            else:
                d = distances[row_idx, neighbors]

                if np.any(d == 0):
                    votes = {
                        cls: np.sum(labels[d == 0] == cls)
                        for cls in self.classes_
                    }
                else:
                    weights = 1.0 / (d + 1e-12)
                    votes = {
                        cls: np.sum(weights[labels == cls])
                        for cls in self.classes_
                    }

            predictions.append(
                max(self.classes_, key=lambda cls: votes[cls])
            )

        return np.asarray(predictions)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


In [ ]:
my_knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", MyKNNClassifier())
])

my_knn_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "model__p": [1, 2],
    "model__weights": ["uniform", "distance"],
}

my_knn_search = GridSearchCV(
    my_knn_pipeline,
    param_grid=my_knn_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

my_knn_search.fit(X_train, y_train)

my_knn_pred = my_knn_search.predict(X_test)

print("Лучшие параметры:", my_knn_search.best_params_)
print("Лучшая CV accuracy:", round(my_knn_search.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_test, my_knn_pred), 4))


# 5. Библиотечные реализации

Сравниваются:

1. Logistic Regression;
2. Support Vector Machine;
3. KNN из `sklearn`;
4. Decision Tree.

Для каждой модели выполняется отдельный подбор гиперпараметров по 5-fold Stratified Cross-Validation.


In [ ]:
models_and_grids = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=5000,
            solver="liblinear",
            random_state=RANDOM_STATE
        ),
        {
            "model__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
        }
    ),

    "SVM": (
        SVC(random_state=RANDOM_STATE),
        {
            "model__C": [0.1, 0.3, 1, 3, 10, 30],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
            "model__min_samples_leaf": [1, 2, 4, 8, 12]
        }
    ),
}


In [ ]:
searches = {}
best_models = {}

for name, (estimator, param_grid) in models_and_grids.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        return_train_score=False
    )

    search.fit(X_train, y_train)

    searches[name] = search
    best_models[name] = search.best_estimator_

    print(f"\n{name}")
    print("Лучшие параметры:", search.best_params_)
    print("Лучшая CV accuracy:", round(search.best_score_, 4))


## 6. Финальная оценка на test

Рассчитываются:

- `accuracy`;
- `precision`;
- `recall`;
- `F1`.

Для медицинской постановки особенно важен `recall` класса `1`, поскольку `FN` соответствует больному пациенту, которого модель отнесла к здоровым.


In [ ]:
all_models = {
    "My KNN": my_knn_search.best_estimator_,
    **best_models
}

results = []
predictions = {}

for name, model in all_models.items():
    pred = model.predict(X_test)
    predictions[name] = pred

    results.append({
        "Model": name,
        "CV accuracy": (
            my_knn_search.best_score_
            if name == "My KNN"
            else searches[name].best_score_
        ),
        "Test accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Test accuracy", ascending=False)
    .reset_index(drop=True)
)

display(results_df.style.format({
    "CV accuracy": "{:.4f}",
    "Test accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
}))


In [ ]:
plot_df = results_df.sort_values("Test accuracy", ascending=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Test accuracy", y="Model")
plt.xlim(0, 1)
plt.xlabel("Test accuracy")
plt.ylabel("")
plt.title("Сравнение моделей")
plt.tight_layout()
plt.show()


## 7. Confusion matrices

Матрица ошибок:

\[
\begin{pmatrix}
TN & FP \\
FN & TP
\end{pmatrix}.
\]

Здесь особенно важны `FN`: это случаи, когда заболевание присутствует, но модель предсказывает класс `0`.


In [ ]:
fig, axes = plt.subplots(1, len(all_models), figsize=(5 * len(all_models), 4))

for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Предсказанный класс")
    ax.set_ylabel("Истинный класс")

plt.tight_layout()
plt.show()


In [ ]:
for name, pred in predictions.items():
    print("=" * 70)
    print(name)
    print(classification_report(
        y_test,
        pred,
        target_names=["Нет заболевания", "Есть заболевание"],
        digits=4
    ))


# 8. Итоговые выводы

- В исходном датасете обнаружены полные дубликаты; после очистки осталось 302 уникальных наблюдения.
- Пропусков нет.
- EDA показал, что наиболее заметную связь с `target` имеют, среди прочего, `cp`, `thalach`, `oldpeak`, `exang`, `ca`, `thal` и `slope`.
- Категориальные признаки закодированы через one-hot encoding.
- Численные признаки стандартизированы.
- Все preprocessing-операции находятся внутри `Pipeline`, поэтому утечки данных при cross-validation нет.
- Собственная реализация KNN поддерживает настройку `n_neighbors`, `p` и `weights`.
- Для всех моделей использована 5-fold Stratified Cross-Validation для выбора гиперпараметров.
- Финальное сравнение выполнено на отдельной test-выборке.
- Для медицинской задачи недостаточно смотреть только на accuracy: необходимо учитывать recall класса `1` и число ложных отрицательных предсказаний `FN`.

Конкретные лучшие параметры и метрики автоматически выводятся выше, поэтому notebook можно повторно запускать при изменении `random_state`, preprocessing или сетки гиперпараметров.


## 9. Математическая справка по собственной реализации KNN

Для объектов \(x,z\in\mathbb{R}^d\):

\[
d_p(x,z)=
\left(
\sum_{j=1}^{d}|x_j-z_j|^p
\right)^{1/p}.
\]

При `weights="uniform"` каждый из \(k\) соседей имеет одинаковый вес.

При `weights="distance"` используется

\[
w_i=\frac{1}{d_i+\varepsilon},
\]

поэтому ближайшие объекты влияют на решение сильнее.


In [ ]:
print("Лучшие параметры собственной реализации KNN:")
print(my_knn_search.best_params_)

for name, search in searches.items():
    print(f"\n{name}:")
    print(search.best_params_)
